In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

In [4]:
from src.preprocessing.dataset_loader import build_dataset_index
from src.preprocessing.splitter import create_stratified_split
from src.preprocessing.dataloaders import (
    create_datasets,
    create_dataloaders
)

In [5]:
train_paths, train_labels = build_dataset_index(
    "../data/raw/train"
)

test_paths, test_labels = build_dataset_index(
    "../data/raw/test"
)

(
    X_train,
    X_val,
    y_train,
    y_val
) = create_stratified_split(
    train_paths,
    train_labels
)

(
    train_dataset,
    val_dataset,
    test_dataset
) = create_datasets(
    X_train,
    y_train,
    X_val,
    y_val,
    test_paths,
    test_labels
)

(
    train_loader,
    val_loader,
    test_loader
) = create_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset,
    batch_size=32
)

In [6]:
import torch

from src.federated.client import FederatedClient

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

client = FederatedClient(
    client_id="client_1",
    train_loader=train_loader,
    device=device
)

In [7]:
loss, acc = client.train(
    epochs=1
)

print(f"Loss : {loss:.4f}")
print(f"Acc  : {acc:.4f}")

0.0023230896331369877 0.9979408383369446
0.009597011841833591 0.8061286211013794
0.008036984130740166 0.9926438927650452
0.01141214556992054 0.8177213072776794
0.003677140921354294 0.994540274143219
0.009116207249462605 0.8684697151184082
0.003770368406549096 0.9898647665977478
0.03543629124760628 0.8803099989891052
0.0010730844223871827 0.9957602620124817
0.04382185637950897 0.8501866459846497
0.001665113610215485 0.9976856708526611
0.04566684737801552 0.8139362931251526
0.002981803845614195 0.9971627593040466
0.042808897793293 0.8861011862754822
0.0011333066504448652 0.9991137385368347
0.056926846504211426 0.8378462791442871
0.00018814172653947026 0.9998389482498169
0.022113289684057236 0.8956459164619446
0.0002718361502047628 0.9995504021644592
0.060556560754776 0.8872706294059753
0.000125777383800596 0.9998832941055298
0.05932244285941124 0.9400220513343811
3.4544958907645196e-05 0.9999759197235107
0.08629794418811798 0.9355788826942444
9.070970554603264e-05 0.9999303817749023
0.04

In [8]:
weights = client.get_weights()

print(type(weights))

<class 'collections.OrderedDict'>


In [9]:
print(len(weights))

41


In [10]:
for key in weights.keys():
    print(key)

features.0.weight
features.1.weight
features.1.bias
features.1.running_mean
features.1.running_var
features.1.num_batches_tracked
features.3.weight
features.4.weight
features.4.bias
features.4.running_mean
features.4.running_var
features.4.num_batches_tracked
features.7.weight
features.8.weight
features.8.bias
features.8.running_mean
features.8.running_var
features.8.num_batches_tracked
features.10.weight
features.11.weight
features.11.bias
features.11.running_mean
features.11.running_var
features.11.num_batches_tracked
features.14.weight
features.15.weight
features.15.bias
features.15.running_mean
features.15.running_var
features.15.num_batches_tracked
features.17.weight
features.18.weight
features.18.bias
features.18.running_mean
features.18.running_var
features.18.num_batches_tracked
cbam.channel_attention.mlp.0.weight
cbam.channel_attention.mlp.2.weight
cbam.spatial_attention.conv.weight
classifier.2.weight
classifier.2.bias


In [11]:
old_weights = client.get_weights()

In [12]:
client2 = FederatedClient(
    client_id="client_2",
    train_loader=train_loader,
    device=device
)

In [13]:
client2.set_weights(
    old_weights
)

In [14]:
weights2 = client2.get_weights()

for key in old_weights:

    if not torch.equal(
        old_weights[key],
        weights2[key]
    ):
        print("Mismatch!")
        break

else:
    print("Weights identical")

Weights identical


In [15]:
before = client.get_weights()

In [16]:
client.train(epochs=1)

8.584489114582539e-05 0.9999812841415405
0.5135414600372314 0.9999661445617676
8.463402627967298e-05 0.9999815225601196
0.48064759373664856 0.9999815225601196
1.1553603144420777e-05 0.9999982118606567
0.5111667513847351 0.999997615814209
6.481643504230306e-05 0.9999868869781494
0.5151078701019287 0.9999955892562866
3.0184848583303392e-05 0.9999945163726807
0.4847574830055237 0.9999865293502808
6.583158665307565e-06 0.9999990463256836
0.5133492946624756 0.9999942779541016
6.006810508552007e-06 0.9999991655349731
0.5259785652160645 0.9999980926513672
4.003104550065473e-06 0.9999995231628418
0.5075258016586304 0.9999454021453857
1.3164063602744136e-05 0.9999978542327881
0.4960940480232239 0.9999685287475586
1.9446943042567e-05 0.9999970197677612
0.5177804827690125 0.9999967813491821
4.757555871037766e-06 0.9999994039535522
0.48855507373809814 0.9999868869781494
4.529936177277705e-06 0.9999992847442627
0.471832811832428 0.999963641166687
1.4725437722518109e-05 0.9999978542327881
0.50141876

(0.28372876579297407, 0.8794529262086513)

In [17]:
after = client.get_weights()

In [ ]:
for key in before:

    if not torch.equal(
        before[key],
        after[key]
    ):
        print("Weights updated")
        break

Weights updated
